In [1]:
!pip install torch==2.2.1 numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 69.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 39.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
!nvidia-smi

Wed Jul 15 07:34:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls /content/drive/MyDrive

'2026_AI Course_Vu Quoc Thai.pdf'
 361.jpg
 Advanced_Deep_Learning_Group_Project.pdf
'Cambridge International AS and A Level Physics Coursebook.pdf'
 Chapter4_Microeconomics_The_production_function_Tutorial.pdf
 Classroom
'Colab Notebooks'
 Combination.gslides
 Docs
'Document sans titre.gdoc'
'Essential Documents'
 Europass_CV3_VuQuocThai.pdf
 FB_IMG_1535004942783.jpg
 FB_IMG_1537101796577.jpg
 Français
 GraphML_Project
 Group_10.pdf
'IELTS_2025_VuQuocThai (1).pdf'
 IELTS_2025_VuQuocThai.pdf
'IELTS Scan'
 Lemuroid
 MeMeMe
 ML.pptx
'Neo4J Report.docx'
'OCR Test.gsheet'
 Olist.drawio
 QC
 Scholar.gdoc
'Seminar 1 Draft.gdoc'
'Seminar 1 - VQT.gdoc'
 Shared
'Souvernirs en images'
'Startup - App Ôsin.gsheet'
'Taiwan - 02.09.2024'
 Takeout
 Thai_GraphAnal.gdoc
 Tickets
'Tinder for Restaurants.gsheet'
'Vu Quoc Thai_1_References_RPI.pdf'


In [5]:
!ls /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/DATA

data.zip  IEEE	    MOOC_0.1  REDDIT_0.1  WIKI_0.1
FDD	  __MACOSX  MOOC_0.3  REDDIT_0.3  WIKI_0.3


In [6]:
%%capture
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html
!python /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/setup.py build_ext --inplace
!python /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/setup.py install
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.1+cu121.html
!pip install torchdiffeq
!pip install torchdata==0.7.1 --no-deps

In [10]:
!cat /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/gen_graph_custom.py 

import argparse
import itertools
import os.path

import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from sklearn.preprocessing import LabelEncoder
import math

parser = argparse.ArgumentParser()
parser.add_argument('--data', type=str, help='dataset name')
parser.add_argument('--raw_data', type=str, help='raw data name')
parser.add_argument('--add_reverse', default=False, action='store_true')
parser.add_argument('--val_ratio', default=0.70, type=float)
parser.add_argument('--test_ratio', default=0.85, type=float)
args = parser.parse_args()

# Processing RAW data
path = "/content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/DATA/IEEE"
raw_df = pd.read_csv('{}/{}/{}.csv'.format(path,args.data, args.raw_data), index_col=0,
                     #names=['src', 'dst', 'time', 'int_roll', 'ext_roll'],
                       header=0)
raw_df['time'] = raw_df['time'] - raw_df['time'].min()
max_t = raw_df['time'].max()
min_t = raw_df['time'].min()
# val_time, test_time = l

In [7]:
!ls /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/DATA/FDD/

edges.csv  ext_full.npz  features.npy  int_full.npz  int_train.npz


In [20]:
%cd /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/
!python gen_graph_custom_Thai.py --raw_data edges --data FDD


/content/drive/.shortcut-targets-by-id/1sjAMkCDHaJ8iKhrDwuTlz87fDSnw4j80/GraphML_Project/Code/GSNOP/code
num_nodes:  120076
100% 1000000/1000000 [00:53<00:00, 18623.74it/s]
100% 120076/120076 [00:00<00:00, 147566.18it/s]
Sorting...
100% 120076/120076 [00:01<00:00, 60967.62it/s]
saving...


In [9]:
# /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code
!python /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/train_np.py --path /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code --data FDD --config /content/drive/MyDrive/GraphML_Project/Code/GSNOP/code/config/JODIE.yml --base_model origin --eval_neg_samples 50

Resize ratio is automatically set to: 10000000
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Epoch 0:
100% 1166/1166 [00:08<00:00, 134.89it/s]
	train loss:1.3811, train nll:1.3811, train kl loss:0.00000  val ap:0.026791  val auc:0.590408 val mrr:0.1287
	total time:8.53s sample time:0.00s prep time:2.57s
Epoch 1:
100% 1166/1166 [00:08<00:00, 133.49it/s]
	train loss:0.8215, train nll:0.8215, train kl loss:0.00000  val ap:0.108282  val auc:0.917571 val mrr:0.3236
	total time:8.62s sample time:0.00s prep time:2.76s
Epoch 2:
100% 1166/1166 [00:07<00:00, 145.87it/s]
	train loss:0.6530, train nll:0.6530, train kl loss:0.00000  val ap:0.109072  val auc:0.917892 val mrr:0.3253
	total time:7.88s sample time:0.00s prep time:2.40s
Epoch 3:
100% 1166/1166 [00:08<00:00, 145.05it/s]
EarlyStopping counter: 1 out of 5
	train loss:0